# nb03 — Synthetic Preservation Probe Generation

Generates ~80 synthetic 1024×1024 16-bit PNG images, each containing 1–2 synthetic
bright streaks on a realistic dark-sky background.  These probes are used as the
**preservation** component of the proxy score: de-poisoned models should still detect
these synthetic genuine streaks with high confidence.

Streak parameters are drawn from the **normal detection distribution** measured in nb02,
NOT from the poison-box distribution — so they are out-of-distribution for the backdoor.

**Outputs saved to `/kaggle/working/probes/`:**
- `*.png` — 16-bit grayscale probe images
- `probes_coco.json` — COCO annotations with one bbox per streak
- `probe_stats.json` — generation parameters for reproducibility

**Rules note:** probes are 100% self-generated from noise + geometry. No test labels used.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import json
import random
import math
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
PROBES_DIR       = Path("/kaggle/working/probes")
PROBES_DIR.mkdir(exist_ok=True)

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

# --- Probe generation parameters (derived from nb02 EDA + normal detection stats) ---
N_PROBES           = 80
SEED               = 42

# Background: dark sky, 16-bit Poisson noise around ~8000
SKY_MEAN_U16       = 8000
SKY_NOISE_SCALE    = 300   # Poisson-like std

# Streak bbox properties — drawn from normal detection distribution, NOT poison
# Normal detection median w=31, h=34, aspect=0.93.  We use a mix of blobs and long streaks.
STREAK_W_RANGE     = (20, 80)   # bbox width  (px)
STREAK_H_RANGE     = (20, 80)   # bbox height (px)
# Actual streak line width inside the bbox (2-4 px)
LINE_WIDTH_RANGE   = (2, 4)
# Peak brightness (16-bit): ~45k/65535 → 4× sky mean (clearly detectable)
STREAK_PEAK_U16    = 45000
STREAK_SIGMA       = 3.0       # Gaussian profile across the line

MARGIN             = 5         # min px from image border

rng = np.random.default_rng(SEED)
print(f"Generating {N_PROBES} probes into {PROBES_DIR}")

## Generate synthetic probe images

In [ ]:
def draw_streak(img_u16, x0, y0, x1, y1, width_px, peak_u16, sigma):
    """
    Draw a bright streak from (x0,y0) to (x1,y1) on a 16-bit grayscale image.
    The streak has a Gaussian cross-section profile.
    Returns the tight axis-aligned bounding box [bx, by, bw, bh] (COCO format).
    """
    dx = x1 - x0
    dy = y1 - y0
    length = math.hypot(dx, dy)
    if length < 1:
        return None
    # Unit normal to the streak direction
    nx = -dy / length
    ny =  dx / length

    # Bounding box with margin equal to 3*sigma to capture the Gaussian tail
    pad = int(sigma * 3) + 1
    bx_min = max(0, min(x0, x1) - pad)
    by_min = max(0, min(y0, y1) - pad)
    bx_max = min(IMG_W, max(x0, x1) + pad)
    by_max = min(IMG_H, max(y0, y1) + pad)

    # Rasterize: for each pixel in bbox, compute distance to the line segment
    ys, xs = np.mgrid[by_min:by_max, bx_min:bx_max]
    px = xs - x0
    py = ys - y0

    # Project onto along-streak and normal directions
    t = (px * dx + py * dy) / (length * length)  # [0,1] along streak
    t_clamped = np.clip(t, 0, 1)
    # Closest point on segment
    cx = x0 + t_clamped * dx
    cy = y0 + t_clamped * dy
    dist = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)

    # Gaussian profile
    gauss = np.exp(-0.5 * (dist / sigma) ** 2)
    # Also taper along-streak ends smoothly
    taper = np.where(t < 0, np.exp(-2 * t ** 2),
             np.where(t > 1, np.exp(-2 * (t - 1) ** 2), 1.0))
    streak_val = (gauss * taper * peak_u16).astype(np.float32)

    # Add (saturate at 65535)
    existing = img_u16[by_min:by_max, bx_min:bx_max].astype(np.float32)
    img_u16[by_min:by_max, bx_min:bx_max] = np.clip(
        existing + streak_val, 0, 65535
    ).astype(np.uint16)

    return [bx_min, by_min, bx_max - bx_min, by_max - by_min]


coco_images      = []
coco_annotations = []
ann_id = 1

for probe_idx in tqdm(range(N_PROBES), desc="Generating probes"):
    # Dark sky background with Poisson-like noise
    background = (rng.normal(SKY_MEAN_U16, SKY_NOISE_SCALE, (IMG_H, IMG_W))
                  .clip(0, 65535).astype(np.uint16))

    # Decide number of streaks per image (1 or 2)
    n_streaks = 1 if probe_idx < N_PROBES * 0.7 else 2
    bboxes_this = []

    for _ in range(n_streaks):
        # Draw bbox dimensions from the normal-detection range
        bw = int(rng.integers(*STREAK_W_RANGE))
        bh = int(rng.integers(*STREAK_H_RANGE))

        # Pick a random angle and compute the streak end-points inside the bbox
        angle = rng.uniform(0, math.pi)  # angle of the streak

        # Centre of the bbox, placed randomly in the image
        cx = int(rng.integers(MARGIN + bw // 2, IMG_W - MARGIN - bw // 2))
        cy = int(rng.integers(MARGIN + bh // 2, IMG_H - MARGIN - bh // 2))

        # Streak length = half-diagonal of bbox
        half_len = math.hypot(bw, bh) / 2
        x0 = int(cx - half_len * math.cos(angle))
        y0 = int(cy - half_len * math.sin(angle))
        x1 = int(cx + half_len * math.cos(angle))
        y1 = int(cy + half_len * math.sin(angle))

        line_w = int(rng.integers(*LINE_WIDTH_RANGE))
        bbox = draw_streak(background, x0, y0, x1, y1,
                           line_w, STREAK_PEAK_U16, STREAK_SIGMA)
        if bbox is not None:
            bboxes_this.append(bbox)

    fname = f"probe_{probe_idx:04d}.png"
    cv2.imwrite(str(PROBES_DIR / fname), background)

    img_id = probe_idx + 1
    coco_images.append({
        "id":        img_id,
        "file_name": fname,
        "height":    IMG_H,
        "width":     IMG_W,
    })

    for bbox in bboxes_this:
        coco_annotations.append({
            "id":         ann_id,
            "image_id":   img_id,
            "category_id": 1,
            "bbox":       [float(v) for v in bbox],
            "area":       float(bbox[2] * bbox[3]),
            "iscrowd":    0,
        })
        ann_id += 1

coco_out = {
    "images":      coco_images,
    "annotations": coco_annotations,
    "categories":  [{"id": 1, "name": "streak", "supercategory": "object"}],
}
with open(PROBES_DIR / "probes_coco.json", "w") as f:
    json.dump(coco_out, f, indent=2)

print(f"Generated {N_PROBES} probe images, {len(coco_annotations)} annotations.")
print(f"Saved to {PROBES_DIR}")

## Verify: poisoned model detects the synthetic streaks

The probes must be realistic enough for the poisoned model to detect them with
solid confidence — otherwise they're not useful preservation probes.
Target: mean confidence >= 0.30, recall >= 50%.

In [ ]:
def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def compute_iou(box_a, box_b):
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    area_a = (box_a[2]-box_a[0]) * (box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0]) * (box_b[3]-box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

cfg_v = get_cfg()
cfg_v.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
cfg_v.MODEL.WEIGHTS                        = POISONED_WEIGHTS
cfg_v.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
cfg_v.MODEL.RETINANET.SCORE_THRESH_TEST    = CONF_THRESH
cfg_v.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg_v.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
predictor_v = DefaultPredictor(cfg_v)

with open(PROBES_DIR / "probes_coco.json") as f:
    coco_v = json.load(f)
img_id_to_anns = {}
for ann in coco_v['annotations']:
    img_id_to_anns.setdefault(ann['image_id'], []).append(ann)

hits = 0
total_anns = 0
confs_detected = []

for im_info in tqdm(coco_v['images'], desc="Verifying probes"):
    img_path = PROBES_DIR / im_info['file_name']
    anns = img_id_to_anns.get(im_info['id'], [])
    im   = load_for_inference(img_path)
    out  = predictor_v(im)['instances'].to('cpu')
    det_boxes = out.pred_boxes.tensor.numpy()
    det_confs = out.scores.numpy()

    for ann in anns:
        total_anns += 1
        bx, by, bw, bh = ann['bbox']
        gt_xyxy = [bx, by, bx + bw, by + bh]
        best_conf = 0.0
        for (x1, y1, x2, y2), dc in zip(det_boxes, det_confs):
            if compute_iou([x1, y1, x2, y2], gt_xyxy) >= 0.2:
                best_conf = max(best_conf, float(dc))
        if best_conf > 0:
            hits += 1
            confs_detected.append(best_conf)

recall = hits / total_anns if total_anns > 0 else 0.0
mean_conf = float(np.mean(confs_detected)) if confs_detected else 0.0

print(f"\nProbe verification results:")
print(f"  Total streak annotations : {total_anns}")
print(f"  Detected (IoU >= 0.2)    : {hits}")
print(f"  Recall @ IoU 0.2         : {recall:.3f}")
print(f"  Mean confidence (hits)   : {mean_conf:.4f}")

if recall >= 0.5 and mean_conf >= 0.30:
    print("\n✓ Probes are realistic: high recall and confidence. Ready for proxy harness.")
else:
    print("\n⚠ Low recall or confidence — probes may not be realistic enough.")
    print("  Consider increasing STREAK_PEAK_U16 or adjusting parameters.")

# Save stats
stats = {
    'n_probes':       N_PROBES,
    'n_annotations':  total_anns,
    'recall_at_iou02': round(recall, 4),
    'mean_conf_hits':  round(mean_conf, 4),
    'sky_mean_u16':   SKY_MEAN_U16,
    'streak_peak_u16': STREAK_PEAK_U16,
    'seed':           SEED,
}
with open(PROBES_DIR / 'probe_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f"\nStats saved to {PROBES_DIR / 'probe_stats.json'}")